# Car Price Prediction with Machine Learning

Predicting the selling price of used cars based on features like brand, age, mileage, fuel type,
and transmission.

**Dataset:** "Vehicle dataset from CarDekho" (Kaggle) — [https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho)

**Libraries used:** `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`


## 1. Setup

In [ ]:
# Core data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", None)


## 2. Data Acquisition

1. Go to the [CarDekho Vehicle Dataset page](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho) on Kaggle.
2. Download and unzip the CSV (commonly named `car data.csv` or `CAR DETAILS FROM CAR DEKHO.csv`).
3. Place it in a `data/` folder in this project, or update `DATASET_PATH` below.

> ℹ️ Column names vary slightly between versions of this dataset (e.g. `Car_Name` vs `name`,
> `Selling_Price` vs `selling_price`). The code below uses the common lowercase-friendly naming;
> if your file uses different capitalization, adjust the `COLUMN_MAP` in the next cell.


In [ ]:
DATASET_PATH = "[DATASET_PATH]"  # e.g. "./data/car data.csv"

df = pd.read_csv(DATASET_PATH)
print("Shape (rows, columns):", df.shape)
df.head()


In [ ]:
# Standardize column names to a consistent lowercase/underscore format,
# regardless of which version of the CarDekho dataset you downloaded.
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Columns:", list(df.columns))

# Common CarDekho column name variants -> the standardized names this notebook expects.
COLUMN_MAP = {
    "car_name": "name",
    "kms_driven": "km_driven",
    "fuel_type": "fuel",
    "seller_type": "seller_type",
}
df = df.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in df.columns})

df.head()


## 3. Data Cleaning

### 3.1 Check for null values and duplicates

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())


In [ ]:
# Drop exact duplicate rows
df = df.drop_duplicates()

# Drop rows missing critical fields (name, year, selling price)
critical_cols = [c for c in ["name", "year", "selling_price"] if c in df.columns]
df = df.dropna(subset=critical_cols)

# For remaining optional columns, fill missing categorical values with "Unknown"
# and missing numeric values with the column median, rather than dropping more rows.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna("Unknown")

for col in df.select_dtypes(include=np.number).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Shape after cleaning:", df.shape)


### 3.2 Fix inconsistent categorical values

Text categories can be entered inconsistently (e.g., `"Petrol"` vs `"petrol"` vs `" Petrol "`).
Standardize casing and whitespace across all categorical columns.


In [ ]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()

for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# Quick check: unique values per categorical column, to catch any remaining inconsistencies
for col in categorical_cols:
    print(f"{col}: {sorted(df[col].unique())}")


## 4. Feature Engineering

### 4.1 Calculate car age from the year column

In [ ]:
CURRENT_YEAR = pd.Timestamp.now().year
df["car_age"] = CURRENT_YEAR - df["year"]

df[["year", "car_age"]].head()


### 4.2 Extract brand from the car name column

The `name` column typically contains the full model name (e.g., `"Maruti Swift Dzire VDI"`).
The brand is usually the first word.


In [ ]:
df["brand"] = df["name"].str.split().str[0].str.title()

print("Number of unique brands:", df["brand"].nunique())
df["brand"].value_counts().head(10)


## 5. Exploratory Data Analysis

### 5.1 Distribution of selling prices

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df["selling_price"], bins=40, kde=True, color="steelblue", ax=ax)
ax.set_title("Distribution of Selling Prices")
ax.set_xlabel("Selling Price")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


**Observation:** *(Fill in after running the cell above.)* Used-car prices are typically
right-skewed — most cars cluster at lower price points, with a long tail of expensive, premium
vehicles. Note whether a log transform of price might help before modeling if the skew is severe.


### 5.2 Price vs. fuel type

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df, x="fuel", y="selling_price", hue="fuel", palette="Set2", legend=False, ax=ax)
ax.set_title("Selling Price by Fuel Type")
ax.set_xlabel("Fuel Type")
ax.set_ylabel("Selling Price")
plt.tight_layout()
plt.show()


**Observation:** *(Fill in after running the cell above.)* Compare median prices and spread
across fuel types. Diesel vehicles often command higher resale prices than petrol vehicles in this
dataset, while categories like CNG/electric tend to have fewer listings and more variable pricing.


### 5.3 Price vs. car age

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(data=df, x="car_age", y="selling_price", alpha=0.5, color="darkorange", ax=ax)
ax.set_title("Selling Price vs. Car Age")
ax.set_xlabel("Car Age (years)")
ax.set_ylabel("Selling Price")
plt.tight_layout()
plt.show()


**Observation:** *(Fill in after running the cell above.)* Expect a negative relationship —
older cars tend to sell for less. Watch for whether the relationship looks linear or more like a
depreciation curve (steep drop early, flattening out for older cars), which would suggest
tree-based models may capture the pattern better than plain linear regression.


## 6. Encode Categorical Variables

Use one-hot encoding for categorical features. This is handled inside a `ColumnTransformer` so it
can be cleanly combined with each model in a single pipeline (avoiding data leakage and keeping
train/test encoding consistent).


In [ ]:
# Columns to use as model features
numeric_features = [c for c in ["car_age", "km_driven", "present_price"] if c in df.columns]
categorical_features = [c for c in ["brand", "fuel", "seller_type", "transmission", "owner"] if c in df.columns]

target = "selling_price"

feature_cols = numeric_features + categorical_features
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 7. Feature Correlation Heatmap

In [ ]:
corr_df = df[numeric_features + [target]].copy()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation Between Numeric Features and Selling Price")
plt.tight_layout()
plt.show()


**Observation:** *(Fill in after running the cell above.)* Look for which numeric features
correlate most strongly (positively or negatively) with selling price — `car_age` is typically
negatively correlated, while `present_price` (original showroom price, if available) is usually
strongly positively correlated.


## 8. Train/Test Split

In [ ]:
X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)


## 9. Model Training

Train two regression models for comparison: **Linear Regression** (a simple baseline) and
**Random Forest Regressor** (a more flexible, non-linear model). Each is wrapped in a pipeline
that handles one-hot encoding internally.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",  # keep numeric features as-is
)

models = {
    "Linear Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]),
    "Random Forest Regressor": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)),
    ]),
    "Gradient Boosting Regressor": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ]),
}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)

print("All models trained.")


## 10. Model Evaluation

Evaluate each model using **MAE** (Mean Absolute Error), **RMSE** (Root Mean Squared Error), and
**R² score**.


In [ ]:
results = []

for name, pipeline in models.items():
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2 Score": r2})

results_df = pd.DataFrame(results).sort_values("R2 Score", ascending=False)
results_df


In [ ]:
fig, ax = plt.subplots()
sns.barplot(data=results_df, x="R2 Score", y="Model", hue="Model", palette="Blues_d",
            legend=False, ax=ax)
ax.set_title("Model Comparison: R² Score")
ax.set_xlabel("R² Score")
plt.tight_layout()
plt.show()


**Observation:** *(Fill in after running the cells above.)* Compare MAE, RMSE, and R² across
models. Tree-based ensembles (Random Forest, Gradient Boosting) typically outperform plain Linear
Regression on this kind of dataset because car pricing has non-linear relationships (e.g., steep
early depreciation) that linear models can't capture as well.


## 11. Feature Importance (Best-Performing Model)

Identify the best model from the evaluation table above, then plot its feature importances (for
tree-based models) to see which features drive price predictions the most.


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = models[best_model_name]
print("Best performing model:", best_model_name)

regressor = best_pipeline.named_steps["regressor"]

if hasattr(regressor, "feature_importances_"):
    # Recover feature names after one-hot encoding
    ohe = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
    ohe_feature_names = list(ohe.get_feature_names_out(categorical_features))
    all_feature_names = ohe_feature_names + numeric_features

    importances = pd.Series(regressor.feature_importances_, index=all_feature_names)
    top_importances = importances.sort_values(ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.barplot(x=top_importances.values, y=top_importances.index, hue=top_importances.index,
                palette="Greens_r", legend=False, ax=ax)
    ax.set_title(f"Top 15 Feature Importances — {best_model_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_model_name} does not expose feature_importances_ "
          f"(this is expected for Linear Regression — inspect .coef_ instead).")


**Observation:** *(Fill in after running the cell above.)* Features like `car_age`,
`present_price` (if available), and specific high-value `brand` categories often dominate the
importance ranking, confirming that age and original price are the strongest drivers of resale
value.


## 12. Summary of Findings

*(Fill in this section after running the full notebook on your data. A few prompts to guide your
write-up:)*

- **Best model:** Which model achieved the lowest MAE/RMSE and highest R², and by how much did it
  outperform the others?
- **Price distribution:** Was the selling price distribution skewed, and did that affect model
  choice or the need for transformations?
- **Fuel type & age effects:** How much did fuel type and car age influence price, based on the
  EDA plots?
- **Most important features:** What did the feature importance chart reveal about what actually
  drives used car prices?
- **Limitations / next steps:** What additional features (e.g., mileage/kmpl, engine size,
  location) might improve the model further?
